# 09 — Aggregation Strategy Analysis

Test multiple aggregation strategies on the trained model (using `best.pt`) without re-running the expensive pipeline.

**Approach**:
1. Load best checkpoint
2. Run pipeline on all 8 target datasets ONCE — cache per-pass scores per dataset
3. Test ~12 aggregation strategies on the cached scores
4. Compare AUROC + AUPRC across strategies
5. Identify best strategy overall and per-dataset

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

os.makedirs('results', exist_ok=True)
os.makedirs('results/per_pass_scores', exist_ok=True)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Callable, Dict, List
import time

from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.pipeline.multipass import MultiPassPipeline
from ms_zerogad.pipeline.aggregation import aggregate_scores_with_breakdown
from ms_zerogad.utils.stable_ops import min_max_normalize
from ms_zerogad.evaluation.metrics import compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Load best checkpoint

In [ ]:
ckpt_path = '/content/drive/MyDrive/Project_GraphML/ms-zerogad/checkpoints/best.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['config']

print(f'Best epoch: {ckpt["best_epoch"]}')
print(f'Best AUROC at training: {ckpt["best_auroc"]:.4f}')
print(f'Total epochs trained: {len(ckpt["history"]["epoch"])}')

pipeline = MultiPassPipeline(
    d_prime=cfg['module1']['d_prime'],
    band_low=cfg['module1']['band_low'],
    band_high=cfg['module1']['band_high'],
    alpha_low=cfg['module1']['alpha_low'],
    alpha_mid=cfg['module1']['alpha_mid'],
    alpha_high=cfg['module1']['alpha_high'],
    k_smoothing=cfg['module2']['k_smoothing'],
    sigma=cfg['module2']['sigma'],
    D_rff=cfg['module2']['D_rff'],
    d_svd=cfg['module2']['d_svd'],
    tau=cfg['module2']['tau'],
    kmeans_max_iter=cfg['module2']['kmeans_max_iter'],
    d_hidden=cfg['module4']['d_hidden'],
    d_latent=cfg['module4']['d_latent'],
    num_encoder_layers=cfg['module4']['num_encoder_layers'],
    num_decoder_layers=cfg['module4']['num_decoder_layers'],
    dropout=cfg['module4']['dropout'],
).to(device)

pipeline.load_state_dict(ckpt['state_dict'])
pipeline.eval()
print('Pipeline loaded.')

## 3. Compute & cache per-pass scores for all target datasets

This is the expensive step — runs the full pipeline. Result is cached to disk so subsequent strategy tests are instant.

In [ ]:
@torch.no_grad()
def get_per_pass_scores(pipeline, X, A, y):
    """Run pipeline once, return raw per-pass scores mapped to original nodes + ground truth."""
    X = X.to(device)
    A = A.to(device)

    scores_list, _, tracker, _ = pipeline(X, A, is_training=False)
    n = X.shape[0]

    breakdown = aggregate_scores_with_breakdown(scores_list, tracker, n)

    return {
        'pass1': breakdown['pass1'].cpu(),
        'pass2': breakdown['pass2'].cpu(),
        'pass3': breakdown['pass3'].cpu(),
        'y': y.cpu(),
        'n': n,
    }


target_names = cfg['datasets']['target']
per_pass_cache = {}

for name in target_names:
    cache_path = f'results/per_pass_scores/{name}.pt'

    if os.path.exists(cache_path):
        print(f'[CACHED] {name}')
        per_pass_cache[name] = torch.load(cache_path, weights_only=False)
        continue

    path = f'/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/{name}.mat'
    if not os.path.exists(path):
        print(f'[SKIP] {name}: file not found')
        continue

    t_start = time.time()
    A_sp, X_sp, y_np = load_graph_dataset(path)
    A = sparse_to_torch_dense(A_sp)
    X = feature_to_torch(X_sp, dense=True)
    y = torch.from_numpy(y_np).long()

    cache_data = get_per_pass_scores(pipeline, X, A, y)
    torch.save(cache_data, cache_path)
    per_pass_cache[name] = cache_data

    elapsed = time.time() - t_start
    print(f'[NEW]    {name}: n={cache_data["n"]}, elapsed={elapsed:.1f}s, saved to {cache_path}')

print(f'\nLoaded scores for {len(per_pass_cache)} datasets.')

## 4. Define aggregation strategies

Each strategy is a function `(p1_norm, p2_norm, p3_norm) -> final_scores` operating on already min-max normalized per-pass scores.

In [ ]:
def normalize_passes(scores_dict):
    """Min-max normalize each pass to [0, 1]."""
    return {
        'p1': min_max_normalize(scores_dict['pass1']),
        'p2': min_max_normalize(scores_dict['pass2']),
        'p3': min_max_normalize(scores_dict['pass3']),
    }


STRATEGIES: Dict[str, Callable] = {
    # Single pass
    'P1_only':    lambda p1, p2, p3: p1,
    'P2_only':    lambda p1, p2, p3: p2,
    'P3_only':    lambda p1, p2, p3: p3,

    # MAX variants
    'MAX_all':    lambda p1, p2, p3: torch.maximum(torch.maximum(p1, p2), p3),  # current
    'MAX_P1P2':   lambda p1, p2, p3: torch.maximum(p1, p2),
    'MAX_P2P3':   lambda p1, p2, p3: torch.maximum(p2, p3),
    'MAX_P1P3':   lambda p1, p2, p3: torch.maximum(p1, p3),

    # MEAN variants
    'MEAN_all':   lambda p1, p2, p3: (p1 + p2 + p3) / 3,
    'MEAN_P1P2':  lambda p1, p2, p3: (p1 + p2) / 2,
    'MEAN_P2P3':  lambda p1, p2, p3: (p2 + p3) / 2,

    # Weighted MEAN
    'W[1,2,1]':   lambda p1, p2, p3: (p1 + 2*p2 + p3) / 4,    # emphasize P2
    'W[2,1,1]':   lambda p1, p2, p3: (2*p1 + p2 + p3) / 4,    # emphasize P1
    'W[1,1,2]':   lambda p1, p2, p3: (p1 + p2 + 2*p3) / 4,    # emphasize P3
    'W[2,2,1]':   lambda p1, p2, p3: (2*p1 + 2*p2 + p3) / 5,  # de-emphasize P3
    'W[3,1,0]':   lambda p1, p2, p3: (3*p1 + p2) / 4,         # mostly P1, a bit P2

    # MIN — anomaly detected by ALL passes
    'MIN_all':    lambda p1, p2, p3: torch.minimum(torch.minimum(p1, p2), p3),

    # MEDIAN — robust to outlier passes
    'MEDIAN':     lambda p1, p2, p3: torch.stack([p1, p2, p3]).median(dim=0).values,
}

print(f'Defined {len(STRATEGIES)} strategies.')

## 5. Apply all strategies to all datasets

In [ ]:
rows = []

for ds_name, scores in per_pass_cache.items():
    norm = normalize_passes(scores)
    p1, p2, p3 = norm['p1'], norm['p2'], norm['p3']
    y = scores['y']

    for strat_name, strat_fn in STRATEGIES.items():
        try:
            final = strat_fn(p1, p2, p3)
            metrics = compute_metrics(final, y)
            rows.append({
                'dataset': ds_name,
                'strategy': strat_name,
                'auroc': metrics['auroc'],
                'auprc': metrics['auprc'],
            })
        except Exception as e:
            print(f'Error: {ds_name} / {strat_name}: {e}')

df = pd.DataFrame(rows)
print(f'Computed {len(df)} (dataset, strategy) pairs.')

## 6. Comparison: AUROC matrix

Rows = strategies, Columns = datasets. Bold the per-dataset max.

In [ ]:
auroc_matrix = df.pivot(index='strategy', columns='dataset', values='auroc')

# Reorder rows by mean AUROC (best at top)
auroc_matrix['mean'] = auroc_matrix.mean(axis=1)
auroc_matrix = auroc_matrix.sort_values('mean', ascending=False)
mean_col = auroc_matrix.pop('mean')
auroc_matrix['mean'] = mean_col

print('AUROC by strategy and dataset (higher = better):')
print('=' * 110)
print(auroc_matrix.to_string(float_format='%.4f'))

In [ ]:
auprc_matrix = df.pivot(index='strategy', columns='dataset', values='auprc')
auprc_matrix['mean'] = auprc_matrix.mean(axis=1)
auprc_matrix = auprc_matrix.sort_values('mean', ascending=False)
mean_col = auprc_matrix.pop('mean')
auprc_matrix['mean'] = mean_col

print('AUPRC by strategy and dataset:')
print('=' * 110)
print(auprc_matrix.to_string(float_format='%.4f'))

## 7. Heatmap visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Drop 'mean' column for the visualization (will show as separate)
matrix_data = auroc_matrix.drop(columns='mean')

im = ax.imshow(matrix_data.values, cmap='RdYlGn', aspect='auto', vmin=0.4, vmax=0.7)
ax.set_xticks(range(len(matrix_data.columns)))
ax.set_xticklabels(matrix_data.columns, rotation=45, ha='right')
ax.set_yticks(range(len(matrix_data.index)))
ax.set_yticklabels(matrix_data.index)

# Annotate each cell
for i in range(len(matrix_data.index)):
    for j in range(len(matrix_data.columns)):
        val = matrix_data.values[i, j]
        # Bold if it's the column max
        col_max = matrix_data.values[:, j].max()
        weight = 'bold' if val == col_max else 'normal'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                color='black', fontsize=9, fontweight=weight)

ax.set_title('AUROC by Aggregation Strategy and Dataset (column max in bold)')
plt.colorbar(im, label='AUROC')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/aggregation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Per-dataset best strategy

In [ ]:
print('Best strategy per dataset (by AUROC):')
print('=' * 70)
best_per_dataset = []
for ds in df['dataset'].unique():
    sub = df[df['dataset'] == ds].sort_values('auroc', ascending=False)
    top = sub.iloc[0]
    second = sub.iloc[1]
    best_per_dataset.append({
        'dataset': ds,
        'best_strategy': top['strategy'],
        'best_auroc': top['auroc'],
        'best_auprc': top['auprc'],
        'second_strategy': second['strategy'],
        'second_auroc': second['auroc'],
        'gap': top['auroc'] - second['auroc'],
    })
    print(f"{ds:<12} | {top['strategy']:<12} AUROC={top['auroc']:.4f} "
          f"(2nd: {second['strategy']:<12} {second['auroc']:.4f}, gap={top['auroc']-second['auroc']:+.4f})")

best_df = pd.DataFrame(best_per_dataset)

## 9. Aggregate-level summary

How many datasets does each strategy "win" or rank top-3?

In [ ]:
from collections import defaultdict

wins = defaultdict(int)
top3 = defaultdict(int)

for ds in df['dataset'].unique():
    sub = df[df['dataset'] == ds].sort_values('auroc', ascending=False)
    wins[sub.iloc[0]['strategy']] += 1
    for s in sub.head(3)['strategy']:
        top3[s] += 1

summary_rows = []
for strat in STRATEGIES.keys():
    sub = df[df['strategy'] == strat]
    summary_rows.append({
        'strategy': strat,
        'mean_auroc': sub['auroc'].mean(),
        'median_auroc': sub['auroc'].median(),
        'min_auroc': sub['auroc'].min(),
        'max_auroc': sub['auroc'].max(),
        'mean_auprc': sub['auprc'].mean(),
        'wins': wins[strat],
        'top3_count': top3[strat],
    })

summary_df = pd.DataFrame(summary_rows).sort_values('mean_auroc', ascending=False)
print('Strategy summary (sorted by mean AUROC):')
print('=' * 110)
print(summary_df.to_string(index=False, float_format='%.4f'))

## 10. Specific comparison: current MAX vs. best alternative

In [ ]:
current = summary_df[summary_df['strategy'] == 'MAX_all'].iloc[0]
best = summary_df.iloc[0]

print(f"Current strategy (MAX_all):")
print(f"  Mean AUROC: {current['mean_auroc']:.4f}")
print(f"  Mean AUPRC: {current['mean_auprc']:.4f}")
print(f"  Wins:        {int(current['wins'])} / {len(per_pass_cache)} datasets")

print(f"\nBest strategy ({best['strategy']}):")
print(f"  Mean AUROC: {best['mean_auroc']:.4f}  ({best['mean_auroc']-current['mean_auroc']:+.4f} vs MAX_all)")
print(f"  Mean AUPRC: {best['mean_auprc']:.4f}  ({best['mean_auprc']-current['mean_auprc']:+.4f} vs MAX_all)")
print(f"  Wins:        {int(best['wins'])} / {len(per_pass_cache)} datasets")

# Per-dataset comparison
print('\nPer-dataset comparison:')
print(f"{'dataset':<12} {'MAX_all':>10} {best['strategy']:>12} {'gain':>10}")
print('-' * 50)
for ds in df['dataset'].unique():
    max_score = df[(df['dataset'] == ds) & (df['strategy'] == 'MAX_all')].iloc[0]['auroc']
    best_score = df[(df['dataset'] == ds) & (df['strategy'] == best['strategy'])].iloc[0]['auroc']
    print(f"{ds:<12} {max_score:>10.4f} {best_score:>12.4f} {best_score-max_score:>+10.4f}")

## 11. Multi-pass value check

Does the multi-pass machinery actually help compared to single-pass (P1_only)?

In [ ]:
p1_only = summary_df[summary_df['strategy'] == 'P1_only'].iloc[0]
best_multipass_strategies = ['MAX_all', 'MEAN_all', 'MEDIAN', 'W[1,2,1]', 'W[1,1,2]', 'W[2,1,1]']

print(f"P1_only baseline:")
print(f"  Mean AUROC: {p1_only['mean_auroc']:.4f}")
print(f"  Mean AUPRC: {p1_only['mean_auprc']:.4f}")

print(f"\nMulti-pass strategies vs P1_only:")
print(f"{'strategy':<12} {'mean_AUROC':>10} {'gain':>10} {'mean_AUPRC':>10} {'gain':>10}")
print('-' * 60)
for strat in best_multipass_strategies:
    row = summary_df[summary_df['strategy'] == strat]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    auroc_gain = row['mean_auroc'] - p1_only['mean_auroc']
    auprc_gain = row['mean_auprc'] - p1_only['mean_auprc']
    marker = '✓' if auroc_gain > 0 else '✗'
    print(f"{strat:<12} {row['mean_auroc']:>10.4f} {auroc_gain:>+10.4f} {row['mean_auprc']:>10.4f} {auprc_gain:>+10.4f}  {marker}")

## 12. Save all results

In [ ]:
df.to_csv('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/aggregation_full.csv', index=False)
summary_df.to_csv('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/aggregation_summary.csv', index=False)
best_df.to_csv('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/aggregation_per_dataset_best.csv', index=False)
auroc_matrix.to_csv('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/aggregation_auroc_matrix.csv')
auprc_matrix.to_csv('/content/drive/MyDrive/Project_GraphML/ms-zerogad/results/aggregation_auprc_matrix.csv')

print('Saved:')
for f in ['aggregation_full.csv', 'aggregation_summary.csv',
         'aggregation_per_dataset_best.csv', 'aggregation_auroc_matrix.csv',
         'aggregation_auprc_matrix.csv', 'aggregation_heatmap.png']:
    path = f'results/{f}'
    size = os.path.getsize(path) / 1024
    print(f'  {path} ({size:.1f} KB)')

## 13. Quick interpretation guide

**If `P1_only` wins or ties top-3**: multi-pass machinery isn't adding value. Consider simplifying architecture.

**If a multi-pass strategy beats P1_only by >0.02 AUROC**: multi-pass IS helping but current MAX is suboptimal. Switch aggregation.

**If best strategy varies wildly across datasets**: aggregation should be data-aware (Phase 2).

**If MEDIAN wins**: noise rejection is the main benefit of multi-pass — passes mostly agree but one is noisy.

**If MEAN beats MAX**: scores are roughly calibrated, averaging reduces noise.

**If MAX wins**: at least one pass has a strong signal but only on different nodes — passes are complementary.